# 🚗 YOLO11n Araç Tespiti Eğitimi (Vehicle Detection Training)
### Bitirme Projesi - Trafik Yönetimi ve Federe Öğrenme Veri Seti

Bu notebook, **`vehicle_dataset_v3_augmented`** veri kümesini kullanarak en güncel **YOLO11 Nano (`yolo11n`)** modelini eğitmek ve test etmek için hazırlanmıştır.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aydin-eng/federated-traffic-learning/blob/main/traffic_vehicle_detection_yolo11.ipynb)


## 1. Donanım ve GPU Kontrolü
Google Colab'de GPU (T4 / V100 / A100) hızlandırıcısının aktif olduğunu doğruluyoruz.


In [ ]:
!nvidia-smi


## 2. Deponun Klonlanması ve Çalışma Ortamı Kurulumu
GitHub üzerindeki veri setimizi ve yapılandırma dosyalarımızı Colab ortamına çekiyoruz.


In [ ]:
import os

# Eğer Colab üzerinde çalışıyorsak repoyu klonla
if not os.path.exists('federated-traffic-learning'):
    !git clone https://github.com/aydin-eng/federated-traffic-learning.git
    %cd federated-traffic-learning
else:
    print('Repo zaten mevcut.')

print('Mevcut çalışma dizini:', os.getcwd())


## 3. Gerekli Kütüphanelerin Yüklenmesi
Ultralytics (YOLO11) ve OpenCV gibi bağımlılıkları yüklüyoruz.


In [ ]:
!pip install -q ultralytics


## 4. Veri Seti ve YAML Konfigürasyonu Doğrulama
`vehicle_dataset_v3_augmented` veri setimizin ve `data.yaml` dosyamızın yollarını kontrol ediyoruz.


In [ ]:
import yaml

yaml_path = 'vehicle_dataset_v3_augmented/data.yaml'
with open(yaml_path, 'r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

print('Veri Konfigürasyonu:')
for k, v in cfg.items():
    print(f' - {k}: {v}')


## 5. YOLO11n Modelinin Başlatılması
Ultralytics üzerinden önceden eğitilmiş hafif **YOLO11n (Nano)** modelini indiriyoruz.


In [ ]:
from ultralytics import YOLO

# Önceden eğitilmiş YOLO11 Nano modelini yükle
model = YOLO('yolo11n.pt')
print('Model başarıyla yüklendi!')
model.info()


## 6. Model Eğitimi (Training) 🚀
Modeli artırılmış eğitim veri kümemiz üzerinde eğitiyoruz.

* **Epochs:** 50 (İhtiyaca göre değiştirebilirsin)
* **Image Size:** 640x640
* **Batch Size:** 16
* **Patience:** 15 (Early Stopping: Belirli süre gelişme olmazsa otomatik durdurur)


In [ ]:
# Eğitimi başlat
results = model.train(
    data='vehicle_dataset_v3_augmented/data.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,                   # Colab GPU
    project='runs/train',
    name='yolo11n_vehicle_exp',
    save=True,
    plots=True,
    patience=15,
    seed=42,
    verbose=True
)


## 7. Model Başarısının Test Kümesinde Değerlendirilmesi (Evaluation)
Eğitilen en iyi ağırlık (`best.pt`) ile test kümesi (`images/test`) üzerindeki mAP50, mAP50-95, Precision ve Recall değerlerini ölçüyoruz.


In [ ]:
# Test kümesi üzerinde değerlendirme
metrics = model.val(split='test')

print('\n' + '=' * 45)
print('  TEST KÜMESİ DEĞERLENDİRME SONUÇLARI')
print('=' * 45)
print(f"mAP@50:     {metrics.box.map50:.4f}")
print(f"mAP@50-95:  {metrics.box.map:.4f}")
print(f"Precision:  {metrics.box.mp:.4f}")
print(f"Recall:     {metrics.box.mr:.4f}")
print('=' * 45)


## 8. Eğitim Metrikleri ve Karışıklık Matrisi (Visualizations)
Kayıp eğrilerini, PR eğrisini ve sınıflara göre Confusion Matrix grafiğini görüntülüyoruz.


In [ ]:
import glob
from IPython.display import Image, display

plots = [
    'runs/train/yolo11n_vehicle_exp/results.png',
    'runs/train/yolo11n_vehicle_exp/confusion_matrix.png',
    'runs/train/yolo11n_vehicle_exp/PR_curve.png'
]

for p in plots:
    if os.path.exists(p):
        print(f'Görsel: {os.path.basename(p)}')
        display(Image(filename=p, width=700))


## 9. Test Görselleri Üzerinde Model Çıkarımı (Inference)
Modelin hiç görmediği test fotoğrafları üzerindeki tespitlerini sınırlayıcı kutular (bounding boxes) ile çizdiriyoruz.


In [ ]:
# Test görselleri üzerinde tahmin yap
test_preds = model.predict(
    source='vehicle_dataset_v3_augmented/images/test',
    conf=0.25,
    save=True,
    project='runs/detect',
    name='test_results'
)

# İlk 4 test sonucunu görselleştir
pred_images = glob.glob('runs/detect/test_results/*.jpg')
print(f'Toplam {len(pred_images)} test görseli işlendi. Örnek sonuçlar:')
for img_path in pred_images[:4]:
    display(Image(filename=img_path, width=650))


## 10. Model Ağırlıklarının Kaydedilmesi
Eğitilen `best.pt` ağırlıklarının konumunu alıyoruz (Federe Öğrenme istemcilerine dağıtılacak temel model).


In [ ]:
best_weights = 'runs/train/yolo11n_vehicle_exp/weights/best.pt'
if os.path.exists(best_weights):
    size_mb = os.path.getsize(best_weights) / (1024 * 1024)
    print(f'En başarılı model ağırlığı hazır: {best_weights} ({size_mb:.2f} MB)')
    print('Bu ağırlığı bilgisayarınıza indirip veya GitHub reponuza ekleyip Federe Öğrenme simülasyonunda kullanabilirsiniz.')
